In [ ]:
from abc import ABC, abstractmethod


class InvalidOrderException(Exception): pass
class InvalidPriceError(Exception): pass
class ItemAlreadyExistsError(Exception): pass


class FoodItem(ABC):
    def __init__(self, item_id, name, price):
        self.__item_id = item_id
        self.__name = name
        if price <= 0:
            raise InvalidPriceError("Price must be greater than 0")
        self.__price = price

    def get_item_id(self): return self.__item_id
    def get_name(self): return self.__name
    def get_price(self): return self.__price

    def set_price(self, price):
        if price <= 0:
            raise InvalidPriceError("Price must be greater than 0")
        self.__price = price

    def set_name(self, name): self.__name = name

    @abstractmethod
    def calculate_total(self, quantity): pass

    @abstractmethod
    def display_info(self): pass

    def place_order(self, quantity):
        try:
            quantity = int(quantity)
            if quantity <= 0:
                raise ValueError(f"Quantity must be greater than 0, got {quantity}")
            total = self.calculate_total(quantity)
            return total
        except ValueError as e:
            raise InvalidOrderException(f"Invalid quantity: {e}") from e
        finally:
            print("--- order attempt done ---")


class VegItem(FoodItem):
    def __init__(self, item_id, name, price, category):
        super().__init__(item_id, name, price)
        self.__category = category

    def get_category(self): return self.__category
    def set_category(self, c): self.__category = c

    def calculate_total(self, quantity):
        return self.get_price() * quantity

    def display_info(self):
        print(f" {self.get_item_id()} | {self.get_name()} | Rs.{self.get_price():.2f} | {self.__category} | Veg")


class NonVegItem(FoodItem):
    EXTRA_CHARGE = 0.10

    def __init__(self, item_id, name, price, category):
        super().__init__(item_id, name, price)
        self.__category = category

    def get_category(self): return self.__category
    def set_category(self, c): self.__category = c

    def calculate_total(self, quantity):
        base = self.get_price() * quantity
        return base + base * NonVegItem.EXTRA_CHARGE

    def display_info(self):
        print(f" {self.get_item_id()} | {self.get_name()} | Rs.{self.get_price():.2f} | {self.__category} | Non-Veg (+10%)")


menu = {}


def add_veg_item():
    try:
        item_id = input("Enter item ID: ")
        if item_id in menu:
            raise ItemAlreadyExistsError("Item ID already exists")
        name = input("Enter name: ")
        price = float(input("Enter price: "))
        category = input("Enter category (Starter/Main/Dessert): ")
        item = VegItem(item_id, name, price, category)
        menu[item_id] = item
        print("Veg item added successfully")
    except (InvalidPriceError, ItemAlreadyExistsError, ValueError) as e:
        print("Error:", e)
    finally:
        print("--- done ---")


def add_non_veg_item():
    try:
        item_id = input("Enter item ID: ")
        if item_id in menu:
            raise ItemAlreadyExistsError("Item ID already exists")
        name = input("Enter name: ")
        price = float(input("Enter price: "))
        category = input("Enter category (Starter/Main/Dessert): ")
        item = NonVegItem(item_id, name, price, category)
        menu[item_id] = item
        print("Non-Veg item added successfully")
    except (InvalidPriceError, ItemAlreadyExistsError, ValueError) as e:
        print("Error:", e)
    finally:
        print("--- done ---")


def display_menu():
    if not menu:
        print("Menu is empty")
        return
    for item in menu.values():
        item.display_info()


def search_item():
    item_id = input("Enter item ID: ")
    if item_id in menu:
        menu[item_id].display_info()
    else:
        print("Item not found")


def place_order():
    try:
        item_id = input("Enter item ID: ")
        if item_id not in menu:
            print("Item not found")
            return
        quantity = int(input("Enter quantity: "))
        total = menu[item_id].place_order(quantity)
        print(f"Item    : {menu[item_id].get_name()}")
        print(f"Qty     : {quantity}")
        print(f"Total   : Rs.{total:.2f}")
    except InvalidOrderException as e:
        print("Order Error:", e)
        if e.__cause__:
            print("Root Cause:", e.__cause__)
    except ValueError:
        print("Enter a valid number")


def update_price():
    try:
        item_id = input("Enter item ID: ")
        if item_id not in menu:
            print("Item not found")
            return
        price = float(input("Enter new price: "))
        menu[item_id].set_price(price)
        print("Price updated successfully")
    except (InvalidPriceError, ValueError) as e:
        print("Error:", e)
    finally:
        print("--- done ---")


def delete_item():
    item_id = input("Enter item ID: ")
    if item_id in menu:
        menu.pop(item_id)
        print("Item deleted")
    else:
        print("Item not found")


def count_items():
    veg = sum(1 for i in menu.values() if isinstance(i, VegItem))
    non_veg = sum(1 for i in menu.values() if isinstance(i, NonVegItem))
    print(f"Total: {len(menu)}  Veg: {veg}  Non-Veg: {non_veg}")


menu['V001'] = VegItem("V001", "Paneer Butter Masala", 220.00, "Main")
menu['V002'] = VegItem("V002", "Veg Spring Roll", 120.00, "Starter")
menu['V003'] = VegItem("V003", "Gulab Jamun", 80.00, "Dessert")
menu['N001'] = NonVegItem("N001", "Chicken Biryani", 320.00, "Main")
menu['N002'] = NonVegItem("N002", "Mutton Seekh Kebab", 260.00, "Starter")


while True:
    print("\n===== Food Ordering System =====")
    print("1. Add Veg Item")
    print("2. Add Non-Veg Item")
    print("3. Display Menu")
    print("4. Search Item")
    print("5. Place Order")
    print("6. Update Price")
    print("7. Delete Item")
    print("8. Count Items")
    print("9. Exit")
    print("================================")

    try:
        choice = int(input("Enter choice: "))
    except ValueError:
        print("Enter a number")
        continue

    if choice == 1: add_veg_item()
    elif choice == 2: add_non_veg_item()
    elif choice == 3: display_menu()
    elif choice == 4: search_item()
    elif choice == 5: place_order()
    elif choice == 6: update_price()
    elif choice == 7: delete_item()
    elif choice == 8: count_items()
    elif choice == 9:
        print("Goodbye!")
        break
    else:
        print("Invalid choice")